# ScavenPred — Complete, sequential and executable research notebook
**Primary method: grouped nested cross-validation with Welch → Pearson → VIF → forward-wrapper feature selection.** This notebook includes every computational stage used in the **current, verified** training, sequence-similarity screening, model selection, and independent evaluation. The analytical functions below are embedded from the publicly verified scripts (not placeholders or old feature lists).

**Scope and provenance.** The notebook begins with the curated 88-descriptor datasets published in the repository. Retrieval of the historical source proteins from UniProt, experimental measurements, and construction of the original descriptor lookup tables cannot be reproduced from the curated matrices alone; they are not reconstructed or invented here. The workbook instead fully reproduces the **reported current modelling and evaluation task**. The original model's historical frozen features are archived separately.

Execute cells **in order** from the repository root or `reproducibility/notebooks`. The nested forward wrapper fits many RF models and may require substantial computing time. It was previously verified independently by GitHub Actions. Unexecuted cells contain no fabricated outputs.

## 0. Imports, reproducible environment, and paths

In [ ]:
from pathlib import Path
import os, sys, io, json, base64, hashlib, tarfile, platform, warnings, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import scipy
from scipy.stats import ttest_ind
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score,
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, classification_report)
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
print("Python:", sys.version.split()[0], "| sklearn:", sklearn.__version__)
assert sklearn.__version__ == "1.8.0", "Install reproducibility/requirements.txt (scikit-learn==1.8.0) for the verified metrics."

paths_to_try = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in paths_to_try if (p/"reproducibility/data/archive_parts").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from a clone of https://github.com/agboken/ScavenPred-ACE-iPred")
REPRO = ROOT/"reproducibility"
DATA = REPRO/"data"/"csv"
OUTPUT = ROOT/"notebook_results"
OUTPUT.mkdir(exist_ok=True)
print("Repository:", ROOT)
MODEL = "scavenpred"
MODEL_OUTPUT = OUTPUT/MODEL
MODEL_OUTPUT.mkdir(parents=True,exist_ok=True)

## 1. Restore and SHA-256 verify the public datasets
The twenty-three published base64 parts are reconstructed into CSV files. Integrity is established against the original SHA-256 checksum. No deprecated root-level Excel files are needed.

In [ ]:
EXPECTED_SHA256 = "5361826554c55de3f0f05b725f6c86eaa0d1c701a985728b28366fc12d6fb6bc"
PARTS = 23

def restore(root: Path, dest: Path):
    source = root / "data" / "archive_parts"
    files = [source / f"part_{i:03}.b64" for i in range(1, PARTS+1)]
    missing = [f.name for f in files if not f.exists()]
    if missing:
        raise FileNotFoundError(f"Missing dataset parts: {missing}")
    text = "".join(f.read_text(encoding="ascii").strip() for f in files)
    data = base64.b64decode(text, validate=True)
    sha = hashlib.sha256(data).hexdigest()
    if sha != EXPECTED_SHA256:
        raise ValueError(f"Checksum mismatch: {sha}")
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tf:
        for member in tf.getmembers():
            if not member.isfile() or "/" in member.name or ".." in Path(member.name).parts:
                raise ValueError(f"Unsafe archive member: {member.name}")
        tf.extractall(dest, filter="data")
    print(f"Restored {len(list(dest.glob('*.csv')))} CSVs in {dest}")
    print("SHA256:", sha)


In [ ]:
restore(REPRO, DATA)
print("Restored dataset files:", len(list(DATA.glob("*.csv"))))
display(pd.DataFrame({"file": [p.name for p in sorted(DATA.glob("*.csv"))]}))

## 2. Dataset loading, label alignment, and the 88 numerical descriptors
ACE-iPred training labels are aligned by sequence; ScavenPred classes are in the training matrix. These functions assert no duplicate sequences, 88 descriptors and binary labels. The vectors are precomputed averages of per-residue descriptor values.

In [ ]:
CONFIG={'scavenpred':('scavenpred_train_88.csv',None),'ace-ipred':('ace_train_88.csv','ace_train_labels.csv')}
def dataset(root,name):
 f,l=CONFIG[name];df=pd.read_csv(root/f); seq=df.seq.astype(str).str.upper().str.strip()
 assert seq.is_unique
 if l:
  labels=pd.read_csv(root/l);labels.seq=labels.seq.astype(str).str.upper().str.strip()
  assert labels.seq.is_unique and len(labels)==len(df)
  y=seq.map(labels.set_index('seq')['class']);assert y.notna().all()
 else:y=df['class']
 X=df.drop(columns=['seq','class','Unnamed: 0','index'],errors='ignore').astype(float)
 assert X.shape[1]==88 and X.notna().all().all() and set(y.unique())=={0,1}
 return seq,X,y.astype(int)
def make_groups(seq):
 ss=seq.tolist();n=len(ss);parent=list(range(n));size=[1]*n
 def root(i):
  while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
  return i
 def link(a,b):
  a,b=root(a),root(b)
  if a!=b:
   if size[a]<size[b]:a,b=b,a
   parent[b]=a;size[a]+=size[b]
 same_len=defaultdict(list)
 for i,s in enumerate(ss):same_len[len(s)].append(i)
 for length,ids in same_len.items():
  if length<5:continue
  tolerance=int(np.floor(length*.2+1e-9))
  for j,i in enumerate(ids):
   for k in ids[j+1:]:
    if sum(c!=d for c,d in zip(ss[i],ss[k]))<=tolerance:link(i,k)
 ix=defaultdict(list)
 for i,s in enumerate(ss):
  if len(s)>=5:ix[s].append(i)
 for j,s in enumerate(ss):
  for length in range(5,len(s)):
   for a in range(len(s)-length+1):
    for i in ix.get(s[a:a+length],[]):link(i,j)
 roots=[root(i) for i in range(n)];mapping={r:j for j,r in enumerate(sorted(set(roots)))}
 return np.array([mapping[r] for r in roots])


In [ ]:
sequence, X, labels = dataset(DATA,MODEL)
assert len(sequence)==1648
assert X.shape == (len(sequence),88)
print("Training samples:",len(sequence),"features:",X.shape[1])
display(pd.DataFrame({"class":labels.value_counts().index,"count":labels.value_counts().values}))
display(X.head())
display(pd.DataFrame({"missing":[int(X.isna().sum().sum())],"duplicate_sequences":[int(sequence.duplicated().sum())]}))

## 3. Exploratory analysis: class proportions, peptide lengths and descriptor distributions
Visualisations use **training data only** and are descriptive. They do not decide which features enter any model.

In [ ]:
fig,ax=plt.subplots(figsize=(5,3.5))
labels.value_counts().sort_index().plot(kind="bar",ax=ax)
ax.set(xlabel="Class (0 putative negative; 1 positive)",ylabel="Peptide count",title="ScavenPred: training class balance")
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(7,4))
for c in (0,1):
    lengths=sequence[labels==c].str.len()
    ax.hist(lengths,bins=np.arange(1, max(sequence.str.len())+2),alpha=.5,label=f"class {c}")
ax.set(xlabel="Residue count",ylabel="Frequency",title="Training peptide length distributions")
ax.legend();plt.tight_layout();plt.show()
print("Descriptor families:",sorted({''.join(ch for ch in f if not ch.isdigit()) for f in X.columns}))
display(X.describe().T.head(12))

## 4. Sequence-relatedness grouping (same-length ≥80% identity or exact substring length ≥5)
The union-find implementation makes groups *transitive*. Grouping does not use activity labels. Group membership is what prevents close sequence relatives crossing the outer training–validation boundary.

In [ ]:
groups = make_groups(sequence)
assert len(groups)==len(sequence)
counts=pd.Series(groups).value_counts()
print("Sequence groups:",counts.size,"| singletons:",int((counts==1).sum()),"| largest group:",counts.max())
fig,ax=plt.subplots(figsize=(6,3.5))
counts.value_counts().sort_index().plot.bar(ax=ax)
ax.set(xlabel="Peptides per group",ylabel="Number of groups",title="ScavenPred: sequence-group sizes")
plt.tight_layout();plt.show()

## 5. Inspect the five outer group-disjoint partitions
For each outer fold, reserve all validation-group peptides before any supervised screening. The remaining four-fifths are the only data available for that fold's Welch/Pearson/VIF analysis.

In [ ]:
outer_splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
outer_splits=list(outer_splitter.split(X,labels.to_numpy(),groups))
split_rows=[]
for i,(tr,va) in enumerate(outer_splits,1):
    assert not (set(groups[tr])&set(groups[va]))
    split_rows.append({"fold":i,"training_peptides":len(tr),"held_out_peptides":len(va),
                       "training_groups":len(set(groups[tr])),"held_out_groups":len(set(groups[va]))})
display(pd.DataFrame(split_rows))

## 6. Feature-screening implementation: Welch, Pearson correlation and VIF
All screening occurs on **each outer training partition only**. Welch retains p<0.05; Pearson retains |r|<0.70 against already selected descriptors; VIF removes descriptors while the maximum VIF is ≥10. The strongest 12 survivors become forward-wrapper candidates.

In [ ]:
def screen(X,y,candidate_limit=12):
    y=np.asarray(y); statistics=[]
    for f in X.columns:
        p=float(ttest_ind(X.loc[y==1,f].to_numpy(), X.loc[y==0,f].to_numpy(), equal_var=False).pvalue)
        corr=float(np.corrcoef(X[f].to_numpy(),y)[0,1])
        if np.isfinite(p) and p<.05 and np.isfinite(corr):
            statistics.append((f,abs(corr),p))
    statistics.sort(key=lambda row:(-row[1],row[0]))
    retained=[]
    for f,_,_ in statistics:
        if all(abs(float(X[f].corr(X[g])))<.70 for g in retained):retained.append(f)
    candidates=retained.copy()
    signal={f:c for f,c,_ in statistics}
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore',category=RuntimeWarning)
        while len(candidates)>1:
            matrix=np.column_stack([np.ones(len(X)),X[candidates].to_numpy(float)])
            vifs=np.asarray([variance_inflation_factor(matrix,i+1) for i in range(len(candidates))])
            largest=np.nanmax(vifs)
            if largest<10:break
            tied=np.where(np.isclose(vifs,largest,rtol=1e-6,atol=1e-8))[0]
            drop=min(tied,key=lambda i:(signal[candidates[i]],candidates[i]))
            candidates.pop(drop)
    return candidates[:candidate_limit],{'welch':len(statistics),'pearson':len(retained),'vif':len(candidates)}

def rf_parameters(name,trees):
    return dict(n_estimators=trees,max_depth=7 if name=='scavenpred' else 10,
                min_samples_leaf=1 if name=='scavenpred' else 4,
                max_features='sqrt' if name=='scavenpred' else 1.0,
                class_weight=None if name=='scavenpred' else 'balanced',
                random_state=42,n_jobs=1)


In [ ]:
tr,va=outer_splits[0]
first_candidates,first_counts=screen(X.iloc[tr].reset_index(drop=True),labels.to_numpy()[tr],candidate_limit=12)
print("First outer fold: screening counts",first_counts)
print("At most 12 candidates:",first_candidates)
assert len(first_candidates)<=12
fig,ax=plt.subplots(figsize=(6,3))
pd.Series(first_counts).reindex(["welch","pearson","vif"]).plot.bar(ax=ax)
ax.set(xlabel="Screening stage",ylabel="Number of descriptors",title="Outer fold 1: training-only screening")
plt.tight_layout();plt.show()
if len(first_candidates)>1:
    fig,ax=plt.subplots(figsize=(7,6))
    c=X.iloc[tr][first_candidates].corr()
    im=ax.imshow(c,vmin=-1,vmax=1,cmap="coolwarm")
    ax.set_xticks(range(len(c)),c.columns,rotation=90)
    ax.set_yticks(range(len(c)),c.columns)
    fig.colorbar(im,ax=ax,label="Pearson r")
    ax.set_title("Outer fold 1: candidate descriptor correlations")
    plt.tight_layout();plt.show()

## 7. MCC-optimised forward-wrapper code
**No old hard-coded features:** The sequential wrapper starts empty and scores each available candidate across three inner *grouped* folds (two training parts, one validation part, rotated). Each proposed addition must improve mean inner MCC. Inner MCC is **for feature choice, not an independent generalisation estimate**. Welch/Pearson/VIF were fitted to the outer training partition before inner CV; the inner wrapper scores are not leakage-free inner performance estimates.

In [ ]:
def evaluate(data,out,name,max_features,trees,candidate_limit):
    seq,X,y=dataset(data,name)
    y=y.to_numpy();groups=make_groups(seq)
    outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
    oof=[];details=[]
    for fold,(tr,va) in enumerate(outer.split(X,y,groups),1):
        assert set(groups[tr]).isdisjoint(set(groups[va]))
        Xt=X.iloc[tr].reset_index(drop=True); yt=y[tr]; gt=groups[tr]
        candidates,counts=screen(Xt,yt,candidate_limit)
        if not candidates:raise RuntimeError('No candidates in outer fold '+str(fold))
        inner=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=42+fold)
        inner_splits=list(inner.split(Xt,yt,gt))
        for it,iv in inner_splits:assert set(gt[it]).isdisjoint(set(gt[iv]))
        selected=[];steps=[];cache={};previous=-np.inf
        def score(features):
            key=tuple(features)
            if key not in cache:
                scores=[]
                for it,iv in inner_splits:
                    model=RandomForestClassifier(**rf_parameters(name,trees))
                    model.fit(Xt.iloc[it][features],yt[it])
                    pred=model.predict(Xt.iloc[iv][features])
                    scores.append(matthews_corrcoef(yt[iv],pred))
                cache[key]=float(np.mean(scores))
            return cache[key]
        for step in range(min(max_features,len(candidates))):
            options=[(score(selected+[f]),f) for f in candidates if f not in selected]
            best,f=max(options,key=lambda z:(z[0],z[1]))
            if selected and best<=previous:break
            selected.append(f);previous=best
            steps.append({'feature_added':f,'inner_group_cv_mcc_selection_score':best})
        model=RandomForestClassifier(**rf_parameters(name,1000 if name=='scavenpred' else 200))
        model.fit(Xt[selected],yt)
        p=model.predict(X.iloc[va][selected])
        proba=model.predict_proba(X.iloc[va][selected])[:,list(model.classes_).index(1)]
        for idx,pred,prob in zip(va,p,proba):
            oof.append({'sequence':seq.iloc[idx],'group':int(groups[idx]),'fold':fold,
                        'actual':int(y[idx]),'prediction':int(pred),'positive_probability':float(prob)})
        details.append({'fold':fold,'training_n':len(tr),'validation_n':len(va),
                        'validation_group_disjoint':True,'screen_counts':counts,
                        'candidate_features':candidates,'selected_features':selected,
                        'selection_steps':steps,
                        'outer_accuracy':float(accuracy_score(y[va],p)),
                        'outer_mcc':float(matthews_corrcoef(y[va],p)),
                        'inner_models_fitted':len(cache)*3,
                        'outer_confusion_matrix':confusion_matrix(y[va],p,labels=[0,1]).tolist()})
        print(name,'fold',fold,'MCC',details[-1]['outer_mcc'],flush=True)
    df=pd.DataFrame(oof)
    assert len(df)==len(X) and df.sequence.is_unique and df.fold.nunique()==5
    report={'model':name,'training_n':len(X),'group_n':len(np.unique(groups)),
            'independent_test_read':False,'candidate_limit':candidate_limit,
            'max_features':max_features,'inner_trees':trees,
            'oof_accuracy':float(accuracy_score(df.actual,df.prediction)),
            'oof_mcc':float(matthews_corrcoef(df.actual,df.prediction)),
            'oof_roc_auc':float(roc_auc_score(df.actual,df.positive_probability)),
            'oof_pr_auc':float(average_precision_score(df.actual,df.positive_probability)),
            'confusion_matrix':confusion_matrix(df.actual,df.prediction,labels=[0,1]).tolist(),
            'folds':details,
            'versions':{'python':platform.python_version(),'sklearn':sklearn.__version__,
                        'numpy':np.__version__,'pandas':pd.__version__},
            'note':'Welch/Pearson/VIF fitted to OUTER training partition. Inner wrapper scores select features and are not independent generalization estimates. Final OOF scores use untouched OUTER validation groups.'}
    out.mkdir(parents=True,exist_ok=True)
    df.to_csv(out/(name+'_wrapper_group_oof.csv'),index=False)
    (out/(name+'_wrapper_group_report.json')).write_text(json.dumps(report,indent=2))
    print(json.dumps({k:report[k] for k in ['model','oof_accuracy','oof_mcc','oof_roc_auc','oof_pr_auc','confusion_matrix']},indent=2))


## 8. Run all five outer folds: grouped nested CV
Each fold refits the complete screening and forward wrapper, then trains an RF on the selected outer-training descriptors. Concatenate only untouched outer-held-out predictions to compute overall accuracy, MCC, ROC-AUC and PR-AUC.

In [ ]:
evaluate(DATA,MODEL_OUTPUT,MODEL,5 if MODEL=="scavenpred" else 10,40,12)
CV_REPORT=json.loads((MODEL_OUTPUT/f"{MODEL}_wrapper_group_report.json").read_text())
OOF=pd.read_csv(MODEL_OUTPUT/f"{MODEL}_wrapper_group_oof.csv")
assert len(OOF)==len(sequence) and OOF["sequence"].is_unique
display(pd.DataFrame([{"accuracy":CV_REPORT["oof_accuracy"],"MCC":CV_REPORT["oof_mcc"],
       "ROC-AUC":CV_REPORT["oof_roc_auc"],"PR-AUC":CV_REPORT["oof_pr_auc"]}]))
print("Confusion matrix [TN FP; FN TP]:",CV_REPORT["confusion_matrix"])
assert abs(CV_REPORT["oof_accuracy"]-0.70145631068)<1e-9
assert abs(CV_REPORT["oof_mcc"]-0.399790720859)<1e-9
assert abs(matthews_corrcoef(OOF.actual,OOF.prediction)-CV_REPORT["oof_mcc"])<1e-12

## 9. Visualise held-out performance, selected features and fold-to-fold stability

In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(OOF.actual,OOF.prediction,labels=[0,1],ax=ax,cmap="Blues",colorbar=False)
ax.set_title("ScavenPred: grouped nested CV confusion matrix")
plt.tight_layout();plt.show()
fpr,tpr,_=roc_curve(OOF.actual,OOF.positive_probability)
fig,ax=plt.subplots(figsize=(5,4));ax.plot(fpr,tpr,label=f"ROC-AUC={roc_auc_score(OOF.actual,OOF.positive_probability):.3f}")
ax.plot([0,1],[0,1],linestyle="--",color="gray");ax.set(xlabel="False positive rate",ylabel="True positive rate",title="Out-of-fold ROC");ax.legend();plt.tight_layout();plt.show()
precision,recall,_=precision_recall_curve(OOF.actual,OOF.positive_probability)
fig,ax=plt.subplots(figsize=(5,4));ax.plot(recall,precision,label=f"PR-AUC={average_precision_score(OOF.actual,OOF.positive_probability):.3f}")
ax.set(xlabel="Recall",ylabel="Precision",title="Out-of-fold precision–recall");ax.legend();plt.tight_layout();plt.show()
folds=pd.DataFrame([{"fold":f["fold"],"accuracy":f["outer_accuracy"],"MCC":f["outer_mcc"],"n":f["validation_n"]} for f in CV_REPORT["folds"]])
display(folds)
fig,ax=plt.subplots(figsize=(6,3));ax.bar(folds.fold.astype(str),folds.MCC);ax.set(xlabel="Outer held-out fold",ylabel="MCC",title="Fold-level MCC (not averaged for the primary metric)");plt.tight_layout();plt.show()
frequency=Counter(k for row in CV_REPORT["folds"] for k in row["selected_features"])
fig,ax=plt.subplots(figsize=(8,4));pd.Series(frequency).sort_values().plot.barh(ax=ax);ax.set(xlabel="Number of outer folds (of five)",title="Descriptor-selection frequency");plt.tight_layout();plt.show()
display(pd.DataFrame([{"fold":row["fold"],"selected_features":", ".join(row["selected_features"])} for row in CV_REPORT["folds"]]))

## 10. Sequence-similarity-screened independent test: exact implementation
The external test is screened by sequence only: equal-length identity ≥80% for sequences of length ≥5, or full substring/superstring match length ≥5. The whole training dataset is used for screening and wrapper selection. The external activity labels are **not used** for feature selection, hyperparameter optimisation or fitting. If a selected descriptor is missing from the external matrix, the script recovers its mean-amino-acid coefficients using training data only.

In [ ]:
def related(a,b):
    if len(a)==len(b) and len(a)>=5:
        if sum(x!=y for x,y in zip(a,b))<=int(np.floor(.2*len(a)+1e-9)):return True
    return min(len(a),len(b))>=5 and (a in b or b in a)

def run(data,out,name,inner_trees=40,candidate_limit=12):
    seq,X,y=dataset(data,name);groups=make_groups(seq);y=y.to_numpy()
    filename='scavenpred_test_88.csv' if name=='scavenpred' else 'ace_test_88.csv'
    test_path=data/filename
    if not test_path.exists() and name=='scavenpred':
        test_path=data.parent/'scavenpred_test_88.csv'
    if not test_path.exists():
        raise FileNotFoundError(f"Missing external descriptor file: {test_path}")
    test=pd.read_csv(test_path)
    test.seq=test.seq.astype(str).str.strip().str.upper()
    assert test.seq.is_unique and set(test.seq).isdisjoint(set(seq))
    candidates,counts=screen(X,y,candidate_limit)
    inner=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=42)
    splits=list(inner.split(X,y,groups))
    cache={};selected=[];steps=[];previous=-np.inf
    def score(features):
        key=tuple(features)
        if key not in cache:
            scores=[]
            for it,iv in splits:
                model=RandomForestClassifier(**rf_parameters(name,inner_trees))
                model.fit(X.iloc[it][features],y[it])
                scores.append(matthews_corrcoef(y[iv],model.predict(X.iloc[iv][features])))
            cache[key]=float(np.mean(scores))
        return cache[key]
    for _ in range(min(5 if name=='scavenpred' else 10,len(candidates))):
        options=[(score(selected+[f]),f) for f in candidates if f not in selected]
        best,f=max(options,key=lambda v:(v[0],v[1]))
        if selected and best<=previous:break
        selected.append(f);previous=best;steps.append({'feature':f,'inner_group_mcc':best})
    # Reconstruct a selected mean-amino-acid descriptor only from training
    # sequences/descriptors, never from external labels.
    for f in selected:
        if f not in test.columns:
            aa='ACDEFGHIKLMNPQRSTVWY'
            A=np.asarray([[s.count(c)/len(s) for c in aa] for s in seq])
            weights,_,rank,_=np.linalg.lstsq(A,X[f].to_numpy(),rcond=None)
            residual=float(np.max(np.abs(A@weights-X[f].to_numpy())))
            assert rank==20 and residual<1e-9,(f,residual)
            B=np.asarray([[s.count(c)/len(s) for c in aa] for s in test.seq])
            test[f]=B@weights
    overlap=np.asarray([any(related(q,t) for t in seq) for q in test.seq])
    kept=~overlap
    model=RandomForestClassifier(**rf_parameters(name,1000 if name=='scavenpred' else 200))
    model.fit(X[selected],y)
    pred=model.predict(test.loc[kept,selected])
    prob=model.predict_proba(test.loc[kept,selected])[:,list(model.classes_).index(1)]
    truth=test.loc[kept,'class'].astype(int).to_numpy()
    out.mkdir(parents=True,exist_ok=True)
    full=pd.DataFrame({'sequence':test.seq,'excluded_related_to_train':overlap,
                       'actual':test['class'].astype(int),'prediction':np.nan,'positive_probability':np.nan})
    full.loc[kept,'prediction']=pred
    full.loc[kept,'positive_probability']=prob
    full.to_csv(out/(name+'_wrapper_external_predictions.csv'),index=False)
    report={'model':name,'training_n':len(X),'external_before_screening':len(test),
            'excluded_related':int(overlap.sum()),'external_evaluated':int(kept.sum()),
            'selected_features':selected,'selection_steps':steps,'screening_counts':counts,
            'accuracy':float(accuracy_score(truth,pred)),
            'mcc':float(matthews_corrcoef(truth,pred)),
            'roc_auc':float(roc_auc_score(truth,prob)),
            'pr_auc':float(average_precision_score(truth,prob)),
            'confusion_matrix':confusion_matrix(truth,pred,labels=[0,1]).tolist(),
            'test_labels_used_in_fitting':False,'sklearn':sklearn.__version__}
    (out/(name+'_wrapper_external_report.json')).write_text(json.dumps(report,indent=2))
    print(name,json.dumps(report,indent=2),flush=True)


## 11. Refit the final model and evaluate the untouched external cohort
This recomputes the full-training final descriptor set rather than manually reusing a frozen list. The result includes an audit column for every excluded external sequence.

In [ ]:
run(DATA,MODEL_OUTPUT,MODEL,inner_trees=40,candidate_limit=12)
EXT_REPORT=json.loads((MODEL_OUTPUT/f"{MODEL}_wrapper_external_report.json").read_text())
EXT=pd.read_csv(MODEL_OUTPUT/f"{MODEL}_wrapper_external_predictions.csv")
print("Selected descriptors for the final full-training model:",EXT_REPORT["selected_features"])
print("Removed by sequence relatedness:",EXT_REPORT["excluded_related"],"of",EXT_REPORT["external_before_screening"])
display(pd.DataFrame([{"external_n":EXT_REPORT["external_evaluated"],"accuracy":EXT_REPORT["accuracy"],"MCC":EXT_REPORT["mcc"],"ROC-AUC":EXT_REPORT["roc_auc"],"PR-AUC":EXT_REPORT["pr_auc"]}]))
assert EXT_REPORT["external_evaluated"]==250
assert abs(EXT_REPORT["accuracy"]-0.784)<1e-9
assert abs(EXT_REPORT["mcc"]-0.438924355585)<1e-9
assert EXT_REPORT["test_labels_used_in_fitting"] is False
display(EXT.head(10))

## 12. Independent test visualisations and verification from individual predictions

In [ ]:
scored=EXT[EXT.excluded_related_to_train==False].dropna(subset=["prediction"]).copy()
scored["prediction"]=scored.prediction.astype(int)
assert len(scored)==EXT_REPORT["external_evaluated"]
assert abs(matthews_corrcoef(scored.actual,scored.prediction)-EXT_REPORT["mcc"])<1e-12
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(scored.actual,scored.prediction,labels=[0,1],ax=ax,cmap="Blues",colorbar=False)
ax.set_title("ScavenPred: screened external confusion matrix");plt.tight_layout();plt.show()
fpr,tpr,_=roc_curve(scored.actual,scored.positive_probability)
fig,ax=plt.subplots(figsize=(5,4));ax.plot(fpr,tpr,label=f"AUC {roc_auc_score(scored.actual,scored.positive_probability):.3f}")
ax.plot([0,1],[0,1],"--",color="gray");ax.set(xlabel="FPR",ylabel="TPR",title="External ROC");ax.legend();plt.tight_layout();plt.show()
pr,rc,_=precision_recall_curve(scored.actual,scored.positive_probability)
fig,ax=plt.subplots(figsize=(5,4));ax.plot(rc,pr,label=f"AP {average_precision_score(scored.actual,scored.positive_probability):.3f}")
ax.set(xlabel="Recall",ylabel="Precision",title="External precision–recall");ax.legend();plt.tight_layout();plt.show()
print(classification_report(scored.actual,scored.prediction,digits=4,zero_division=0))
print("Confirmed: pooled CV MCC and external MCC both independently match prediction files.")

## 13. Model interpretation: RF feature importance and optional SHAP
The final selected feature subset is recalculated from the actual external training-only wrapper result. RF impurity-based importances and SHAP contributions describe prediction behaviour, **not causal mechanisms or experimentally validated bioactivity determinants**. To avoid unnecessary runtime, SHAP is optional.

In [ ]:
selected=EXT_REPORT["selected_features"]
final_model=RandomForestClassifier(**rf_parameters(MODEL,1000 if MODEL=="scavenpred" else 200))
final_model.fit(X[selected],labels.to_numpy())
imp=pd.Series(final_model.feature_importances_,index=selected).sort_values()
display(imp.rename("RF Gini importance").to_frame())
fig,ax=plt.subplots(figsize=(7,3.5));imp.plot.barh(ax=ax);ax.set(xlabel="Mean decrease in impurity",title="Final model: RF descriptor importance");plt.tight_layout();plt.show()

In [ ]:
RUN_SHAP=False  # Change to True to calculate explanatory values on up to 200 training peptides
if RUN_SHAP:
    import shap
    subset=X[selected].sample(n=min(200,len(X)),random_state=42)
    explainer=shap.TreeExplainer(final_model)
    values=explainer.shap_values(subset)
    if isinstance(values,list):
        sv=np.asarray(values[1])
    else:
        arr=np.asarray(values)
        sv=arr[:,:,1] if arr.ndim==3 else arr
    mean_abs=pd.Series(np.abs(sv).mean(axis=0),index=selected).sort_values()
    display(mean_abs.rename("mean |SHAP|").to_frame())
    fig,ax=plt.subplots(figsize=(7,3.5))
    mean_abs.plot.barh(ax=ax);ax.set(xlabel="Mean absolute SHAP value",title="Final model: descriptor contribution")
    plt.tight_layout();plt.show()

## 14. Save a machine-readable audit summary and figures
The code below writes only derived analysis files under `notebook_results/`. Source input and checked-in model scripts are never overwritten.

In [ ]:
summary={"model":MODEL,"train_n":len(sequence),"descriptor_count":X.shape[1],
  "related_sequence_groups":len(set(groups)),
  "cv":{k:CV_REPORT[k] for k in ["oof_accuracy","oof_mcc","oof_roc_auc","oof_pr_auc","confusion_matrix"]},
  "external":{k:EXT_REPORT[k] for k in ["external_evaluated","excluded_related","selected_features","accuracy","mcc","roc_auc","pr_auc","confusion_matrix"]},
  "scikit_learn":sklearn.__version__}
summary_path=MODEL_OUTPUT/"notebook_audit_summary.json"
summary_path.write_text(json.dumps(summary,indent=2))
print("Saved:",summary_path)
print("CV predictions:",MODEL_OUTPUT/f"{MODEL}_wrapper_group_oof.csv")
print("External predictions:",MODEL_OUTPUT/f"{MODEL}_wrapper_external_predictions.csv")

## 15. Interpretation limits and reproducibility notes
- All **outer-held-out predictions** were made without screening or fitting on the corresponding outer validation peptides or their grouped relatives.
- The **inner wrapper** uses candidates screened on the full outer training partition, so its inner mean MCC is a selection score, not a strictly independent inner estimate.
- Closely related sequences are defined by the published exact identity/substring rules; shorter or more distant homologues can remain separate.
- Grouped CV and screened external results use the **same feature-selection algorithm**, but selected descriptors can differ by outer fold.
- The putative negative labels are not experimentally confirmed inactive; fragmentation of UniProt proteins was intended to avoid imposing one protease's cleavage rule. K-mer removal was a distinct sequence-similarity/redundancy measure.
- Previous fixed-panel notebooks remain under `reproducibility/notebooks/archive/`. Live web predictors were not automatically replaced with these revised fitted models.
- The raw UniProt harvesting/protein fragmentation and original descriptor-scale derivation are upstream data-curation stages not reconstructed here, because the curated repository provides their outcomes rather than complete upstream source records.

Current reports: [grouped wrapper CV](../verification/2026-11-10_grouped_nested_forward_wrapper.md) and [screened external evaluation](../verification/2026-11-10_grouped_forward_wrapper_external.md).